# Analysis

In [ ]:
import cv2
import pandas as pd
from pathlib import Path
import ast

train = pd.read_csv("../data/raw/labels/train.csv")
train.head()

,filename,class,bbox
0,img086351.png,smart_1,"[183, 311, 657, 415]"
1,img083734.png,smart_1,"[361, 352, 479, 716]"
2,img081317.png,smart_1,"[163, 16, 905, 785]"
3,img089140.png,smart_1,"[188, 427, 499, 573]"
4,img082379.png,smart_1,"[152, 462, 1024, 605]"


In [2]:
train.shape

(66000, 3)

In [3]:
train["class"].unique()

<StringArray>
[                'smart_1',                  'cheops',
         'lisa_pathfinder',                  'debris',
             'proba_3_ocs',             'proba_3_csc',
                    'soho', 'earth_observation_sat_1',
                 'proba_2',              'xmm_newton',
             'double_star']
Length: 11, dtype: str

In [4]:
print(train["class"].nunique())

11


In [5]:
img = cv2.imread("../data/raw/train/img000001.jpg")
img.shape

(1024, 1024, 3)

# All classes

In [6]:
classes = sorted(train["class"].unique())
class_to_id = {
    cls: idx
    for idx, cls in enumerate(classes)
}
class_to_id


{'cheops': 0,
 'debris': 1,
 'double_star': 2,
 'earth_observation_sat_1': 3,
 'lisa_pathfinder': 4,
 'proba_2': 5,
 'proba_3_csc': 6,
 'proba_3_ocs': 7,
 'smart_1': 8,
 'soho': 9,
 'xmm_newton': 10}

# New Directory For Train Labels

In [7]:
train_output_dir = Path("../data/clean/labels/train")
train_output_dir.mkdir(
    parents= True,
    exist_ok= True
)

parents=True — создает всю цепочку папок. Если родительских папок ../data/, clean/ или labels/ еще не существует, Python создаст их автоматически. Без этого параметра код выдаст ошибку, если хотя бы одной промежуточной папки нет.

exist_ok=True — игнорирует ошибку, если папка уже создана. По умолчанию Python выдает ошибку, если папка train уже существует. Этот параметр позволяет коду работать дальше без сбоев при повторном запуске.

# Demo Convert Function

In [8]:
bbox = ast.literal_eval(train.iloc[0]["bbox"])
bbox

[183, 311, 657, 415]

In [9]:
x_min, y_min, x_max, y_max = bbox
x_center = ((x_max + x_min) / 2) / 1024
y_center = ((y_max + y_min) / 2) / 1024

width = (x_max - x_min) / 1024
height = (y_max - y_min) / 1024
print(x_center, y_center, width, height)

0.41015625 0.3544921875 0.462890625 0.1015625


# Final Convert Function

In [10]:
IMG_SIZE = 1024

def convert_bbox(bbox_str):
    x_min, y_min, x_max, y_max = ast.literal_eval(bbox_str)

    x_center = ((x_min + x_max) / 2) / IMG_SIZE
    y_center = ((y_min + y_max) / 2) / IMG_SIZE

    width = (x_max - x_min) / IMG_SIZE
    height = (y_max - y_min) / IMG_SIZE

    return x_center, y_center, width, height

In [11]:
convert_bbox(train.iloc[0]["bbox"])

(0.41015625, 0.3544921875, 0.462890625, 0.1015625)

# Train Part

In [ ]:
for _, row in train.iterrows():

    filename = row["filename"]
    class_name = row["class"]

    class_id = class_to_id[class_name]

    x_center, y_center, width, height = convert_bbox(row["bbox"])

    txt_name = Path(filename).stem + ".txt"

    txt_path = train_output_dir / txt_name

    with open(txt_path, "w") as f:
        f.write(
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{width:.6f} "
            f"{height:.6f}"
        )

.stem — берет чистое имя файла (например, из data.csv получится data).


,+ ".txt" — добавляет к этому имени расширение .txt

txt_name = "img086351.txt"

txt_path = output_dir / txt_name

оператор / означает:

"присоедини эту часть пути"

То есть получится:

../data/clean/labels/train/img086351.txt

# Train .txt Check

In [14]:
with open("../data/clean/labels/train/img000001.txt") as f:
    print(f.read())

5 0.441406 0.625977 0.646484 0.748047


In [27]:


len(list(train_output_dir.glob("*.txt")))

66000

In [29]:
len(train)

66000

# Validation Part

In [17]:
val = pd.read_csv("../data/raw/labels/val.csv")
val.head()

,filename,class,bbox
0,img055517.png,lisa_pathfinder,"[375, 273, 610, 496]"
1,img056814.png,lisa_pathfinder,"[678, 302, 811, 448]"
2,img053102.png,lisa_pathfinder,"[246, 247, 466, 473]"
3,img053516.png,lisa_pathfinder,"[338, 600, 601, 878]"
4,img055023.png,lisa_pathfinder,"[545, 408, 762, 596]"


In [18]:
val.shape

(22000, 3)

In [19]:
val['class'].unique()

<StringArray>
[        'lisa_pathfinder',             'proba_3_csc',
                 'smart_1',              'xmm_newton',
                    'soho', 'earth_observation_sat_1',
                  'debris',                 'proba_2',
             'proba_3_ocs',                  'cheops',
             'double_star']
Length: 11, dtype: str

In [20]:
val_output_dir = Path("../data/clean/labels/val")
val_output_dir.mkdir(parents= True, exist_ok= True)

In [24]:
for _, row in val.iterrows():
    
    filename = row["filename"]
    x_center, y_center, width, height = convert_bbox(row["bbox"])
    class_name = row["class"]
    class_id = class_to_id[class_name]

    txt_name = Path(filename).stem + ".txt"
    txt_path = val_output_dir / txt_name

    with open(txt_path, "w") as f:
        f.write(
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{width:.6f} "
            f"{height:.6f}"
        )

In [25]:
len(list(val_output_dir.glob("*.txt")))

22000

In [26]:
len(val)

22000